# Hao et al. (PLOS ONE 20(12): e0332887, 2025): paper-only implementation
*Post-quantum-inspired scalable blockchain architecture for internet hospital systems with lightweight privacy-preserving access control*

This notebook implements **only what the paper describes**: Algorithms 1-5, Eqs. 1-17, the Table 2 parameters and the Sec. 6 experiments.
It contains nothing beyond the paper (no real Kyber, no real ZKP). The paper's "Kyber-inspired" encryption is a mock (random token + AES-GCM)
and its "ZKP" is a SHA-256 hash commitment. Both are implemented exactly that way.

Where the paper leaves a value or formula unspecified, the code uses a labelled assumption (marked `# ASSUMPTION`, listed at the end).
Run: *Runtime > Run all*.

In [ ]:
!pip -q install pycryptodome

In [ ]:
import os, time, hmac, hashlib, json, random, warnings
from dataclasses import dataclass
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from Crypto.PublicKey import RSA
from Crypto.Cipher import PKCS1_OAEP, AES
from Crypto.Random import get_random_bytes

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 170)
rng = np.random.default_rng(42)

CFG = dict(
    # --- Table 2 / Sec 5.2 (from the paper) ---
    N_PATIENTS=1000, N_SENSORS=5, N_STEPS=200,           # 10 min, 1 reading / 3 s
    SIGMA=0.5, SPOOF=(15, 25), ACC_TOL=1.0,
    FAULT_RATES=[0.05, 0.10, 0.20, 0.30],
    CRYPTO_ROUNDS=100, AUTH_ROUNDS=100,
    RAFT_SIZES=[10, 50, 100, 200, 500, 1000], PBFT_SIZES=[4, 10, 20],  # PBFT simulated only up to 20 nodes
    BLOCK_SIZE=20,                                        # 10-20 tx per block
    E_CRYPTO_UJ=30.2, E_MSG_UJ=16.5,                      # Sec 4.6 energy constants
    # --- NOT given in the paper (ASSUMPTION) ---
    P_MISSING=0.3,        # share of injected faults that are dropouts (rest are spoofs)
    EPS=1.5,              # Algorithm 1/2 outlier tolerance (3 sigma)
    HOP_DELAY_MS=1.0,     # constant network delay (paper: "synchronous constant-delay")
    MSG_COST_MS=0.05,     # per-message processing cost in the consensus cost model
    E_RETX_UJ=30.2,       # energy per retransmission (e_re in Eq. 13)
)
RESULTS = {}

## 1. Fault-tolerant sensor ingestion (Sec. 4.1, 5.2, 5.3, 6.8, 6.9)

In [ ]:
def generate_signals(n_pat, n_steps, rng):
    # ASSUMPTION: the paper only says "true physiological signal x*"; baseline + slow random walk
    base = rng.uniform(60, 100, size=(n_pat, 1))
    return base + np.cumsum(rng.normal(0, 0.05, size=(n_pat, n_steps)), axis=1)

def inject_faults(truth, n_sensors, fault_rate, rng):
    """Eq. 2 / Eq. 14: x_i = x* + delta_i + eta_i; delta ~ N(0, sigma), eta = +-15/+-25 spoof, or NaN for missing."""
    P, T = truth.shape; shape = (P, T, n_sensors)
    x = truth[:, :, None] + rng.normal(0, CFG["SIGMA"], shape)
    faulty = rng.random(shape) < fault_rate
    missing = faulty & (rng.random(shape) < CFG["P_MISSING"])
    spoofed = faulty & ~missing
    x = np.where(spoofed, x + rng.choice(CFG["SPOOF"], size=shape) * rng.choice([-1, 1], size=shape), x)
    return np.where(missing, np.nan, x), faulty

def algorithm1(x, eps=CFG["EPS"]):
    """Algorithm 1: adaptive median-based correction. Returns (x_hat, unreliable_flag)."""
    n = x.shape[2]
    x_hat = np.nanmedian(x, axis=2)                                   # X[floor(n/2)] of the sorted readings (NaNs excluded)
    is_faulty = np.isnan(x) | (np.abs(x - x_hat[..., None]) > eps)    # mark x_i faulty if |x_i - x_hat| > eps
    unreliable = is_faulty.sum(axis=2) > n / 2                        # warn: unreliable consensus
    return x_hat, unreliable

def agg_median(x):   return np.nanmedian(x, axis=2)                   # Eq. 1 / Eq. 15
def agg_mean(x):     return np.nanmean(x, axis=2)
def agg_none(x):     return x[:, :, 0]

def agg_majority(x):
    r = np.round(x)
    counts = (r[..., :, None] == r[..., None, :]).sum(-1).astype(float); counts[np.isnan(r)] = -1
    return np.take_along_axis(r, counts.argmax(-1)[..., None], -1)[..., 0]

def algorithm2(x, eps=CFG["EPS"]):
    """Algorithm 2: dynamic aggregation: mean if eta<=0.2, median if 0.2<eta<0.4, else mode of rounded values."""
    med = np.nanmedian(x, axis=2)
    eta = np.where(np.isnan(x), 1.0, (np.abs(x - med[..., None]) > eps).astype(float)).mean(axis=2)   # fault estimate
    return np.where(eta <= 0.2, np.nanmean(x, axis=2), np.where(eta < 0.4, med, agg_majority(x)))

def agg_kalman(x, q=0.05**2, r=0.3**2, gate=5.0):
    # ASSUMPTION: the paper benchmarks a "Kalman filter" without details; scalar filter on the median-fused signal with gating
    z = np.nanmedian(x, axis=2); P_, T = z.shape
    xk = np.where(np.isnan(z[:, 0]), 75.0, z[:, 0]); pk = np.ones(P_); est = np.empty_like(z)
    for t in range(T):
        pk = pk + q; innov = z[:, t] - xk; S = pk + r
        ok = ~np.isnan(innov) & (np.abs(innov) <= gate * np.sqrt(S)); K = pk / S
        xk = np.where(ok, xk + K * innov, xk); pk = np.where(ok, (1 - K) * pk, pk); est[:, t] = xk
    return est

AGGS = {"No correction": agg_none, "Mean smoothing": agg_mean, "Majority voting": agg_majority,
        "3-of-5 Median (Alg. 1)": lambda x: algorithm1(x)[0], "Kalman filter": agg_kalman, "Adaptive (Alg. 2)": algorithm2}

def sensor_accuracy(est, truth):                                      # Eq. 16
    return float(np.mean(np.abs(est - truth) <= CFG["ACC_TOL"]) * 100)

truth = generate_signals(CFG["N_PATIENTS"], CFG["N_STEPS"], rng)
rows, flag_rows = [], []
for fr in CFG["FAULT_RATES"]:
    x, faulty = inject_faults(truth, CFG["N_SENSORS"], fr, rng)
    for name, fn in AGGS.items():
        rows.append(dict(fault_rate=int(fr * 100), method=name, accuracy=sensor_accuracy(fn(x), truth)))
    xh, unrel = algorithm1(x)
    wrong = np.abs(xh - truth) > CFG["ACC_TOL"]
    flag_rows.append(dict(fault_rate=int(fr * 100), flagged_unreliable_pct=unrel.mean() * 100,
                          wrong_pct=wrong.mean() * 100, wrong_that_were_flagged_pct=(unrel & wrong).sum() / max(wrong.sum(), 1) * 100))
sensor_tbl = pd.DataFrame(rows).pivot(index="method", columns="fault_rate", values="accuracy").round(1)
print("Sensor correction accuracy (%) vs. injected fault rate (%)  [Fig 10 / Fig 11]"); display(sensor_tbl)
print("Algorithm 1 'unreliable consensus' warnings:"); display(pd.DataFrame(flag_rows).round(2).set_index("fault_rate"))
RESULTS["median_30"] = sensor_tbl.loc["3-of-5 Median (Alg. 1)", 30]; RESULTS["kalman_30"] = sensor_tbl.loc["Kalman filter", 30]

fig, ax = plt.subplots(figsize=(7, 4))
for m in sensor_tbl.index: ax.plot(sensor_tbl.columns, sensor_tbl.loc[m], marker="o", label=m)
ax.set_xlabel("Fault rate (%)"); ax.set_ylabel("Accuracy within ±1 (%)"); ax.grid(alpha=.3); ax.legend(fontsize=7); ax.set_title("Sensor correction accuracy (cf. Fig 11)"); plt.show()

## 2. Encryption layer (Sec. 4.2, 5.4, 6.5-6.7)
RSA-2048 baseline vs. the paper's **Kyber-768-inspired mock**: a randomised public token derives a symmetric key that encrypts with AES-GCM
(Table 2: "Mock Kyber-768 (AES-GCM with public token simulation)"). Eqs. 3-4.

In [ ]:
class RSAScheme:
    name = "RSA-2048"
    def __init__(self):
        k = RSA.generate(2048); self._e = PKCS1_OAEP.new(k.publickey()); self._d = PKCS1_OAEP.new(k)
    def encrypt(self, m): return self._e.encrypt(m)                   # c = Enc_pk(m)
    def decrypt(self, c): return self._d.decrypt(c)                   # m = Dec_sk(c)

class MockKyberHybrid:
    """Kyber-inspired hybrid *simulation* (as in the paper): public token -> symmetric key -> AES-GCM. Not real Kyber."""
    name = "Kyber-inspired (mock)"
    def __init__(self): self.pk = get_random_bytes(32)                # ASSUMPTION: sk == pk, it only models cost
    def _key(self, token): return hashlib.sha256(self.pk + token).digest()   # 256-bit symmetric key (Table 2)
    def encrypt(self, m):
        token, nonce = get_random_bytes(32), get_random_bytes(12)
        ct, tag = AES.new(self._key(token), AES.MODE_GCM, nonce=nonce).encrypt_and_digest(m)
        return token + nonce + tag + ct
    def decrypt(self, c):
        token, nonce, tag, ct = c[:32], c[32:44], c[44:60], c[60:]
        return AES.new(self._key(token), AES.MODE_GCM, nonce=nonce).decrypt_and_verify(ct, tag)

def bench_scheme(scheme, rounds=CFG["CRYPTO_ROUNDS"]):
    import tracemalloc
    scheme.decrypt(scheme.encrypt(b"warm-up")); enc, dec, size = [], [], []
    for _ in range(rounds):
        m = get_random_bytes(int(rng.integers(64, 129)))               # 64-128 B simulated EHR
        t0 = time.perf_counter_ns(); c = scheme.encrypt(m); t1 = time.perf_counter_ns(); m2 = scheme.decrypt(c); t2 = time.perf_counter_ns()
        assert m2 == m; enc.append((t1 - t0) / 1e6); dec.append((t2 - t1) / 1e6); size.append(len(c))
    tracemalloc.start(); scheme.encrypt(m); peak = tracemalloc.get_traced_memory()[1]; tracemalloc.stop()
    ci = lambda a: 1.96 * np.std(a, ddof=1) / np.sqrt(len(a))          # 95% CI half-width (Sec 5.6 / 6.11)
    return dict(scheme=scheme.name, enc_ms=np.mean(enc), enc_ci95=ci(enc), dec_ms=np.mean(dec), dec_ci95=ci(dec),
                ct_bytes=np.mean(size), peak_mem_B=peak), np.array(enc)

schemes = [RSAScheme(), MockKyberHybrid()]; rows, enc_series = [], {}
for s in schemes: r, e = bench_scheme(s); rows.append(r); enc_series[s.name] = e
crypto_df = pd.DataFrame(rows).set_index("scheme"); display(crypto_df.round(4))
rsa, mock = crypto_df.iloc[0], crypto_df.iloc[1]
RESULTS.update(enc_red=(1 - mock.enc_ms / rsa.enc_ms) * 100, dec_red=(1 - mock.dec_ms / rsa.dec_ms) * 100,
               ct_mock=mock.ct_bytes, ct_rsa=rsa.ct_bytes, mem_red=(1 - mock.peak_mem_B / rsa.peak_mem_B) * 100)
fig, ax = plt.subplots(figsize=(7, 3.5))
for n, e in enc_series.items(): ax.plot(e, label=n, lw=1)
ax.set_yscale("log"); ax.set_xlabel("Sample index"); ax.set_ylabel("Time (ms)"); ax.legend(); ax.grid(alpha=.3)
ax.set_title("Encryption time, 100 rounds (cf. Fig 7)"); plt.show()

## 3. Lightweight privacy-preserving access control (Sec. 4.3, 6.1)
Implements Eqs. 5-10 and Algorithms 3-4: hash commitment `p = H(s)` with `s = HMAC(patientID ‖ permission)`, timestamp + nonce replay protection,
bounded retry, HMAC-committed policy tuples `π = ⟨e, r, R, a, t_e⟩`, latest-expiry conflict resolution, revocation and an audit trail. Baseline: smart-contract-style RBAC.

In [ ]:
MASTER_KEY = get_random_bytes(32)
H = lambda b: hashlib.sha256(b).digest()

class HashCommitmentAC:
    """Algorithm 3 (enrol + verify), Sec 4.3.2 (freshness, grant, revoke, audit), Algorithm 4 (retry)."""
    def __init__(self, window_s=30.0):
        self.commit, self.version, self.seen, self.window, self.audit = {}, {}, set(), window_s, []
    def _secret(self, pid, permission):                                # Eq. 5   (ASSUMPTION: version counter so re-generation yields a new s)
        v = self.version.get(pid, 0)
        return hmac.new(MASTER_KEY, f"{pid}|{permission}|v{v}".encode(), hashlib.sha256).digest()
    def enroll(self, pid, permission):                                 # Alg. 3: p <- H(s); store p on ledger
        s = self._secret(pid, permission); self.commit[pid] = H(s); self.audit.append(("grant", pid, time.time())); return s
    def revoke_and_reissue(self, pid, permission):                     # token revoke: new s' overwrites old p
        self.version[pid] = self.version.get(pid, 0) + 1; self.audit.append(("revoke", pid, time.time())); return self.enroll(pid, permission)
    def make_request(self, pid, s): return dict(pid=pid, s=s, nonce=os.urandom(16).hex(), ts=time.time())
    def verify(self, req, now=None):                                   # Alg. 3 verification + timestamp window + nonce uniqueness
        now = now or time.time()
        if abs(now - req["ts"]) > self.window or req["nonce"] in self.seen: self.audit.append(("deny", req["pid"], now)); return False
        ok = H(req["s"]) == self.commit.get(req["pid"])               # Eq. 7: Verify(p, s') = 1 iff H(s') = p
        if ok: self.seen.add(req["nonce"])
        self.audit.append(("allow" if ok else "deny", req["pid"], now)); return ok
    def verify_with_retry(self, pid, attempts, k=3):                   # Algorithm 4
        d = 0
        for i, s_try in enumerate(attempts):
            if i >= k: break
            if H(s_try) == self.commit.get(pid): d = 1; break
        return d

class PolicyStore:
    """Eqs. 8-10: policy tuples hashed with HMAC under a patient-local key; most recent expiry wins; revocation replaces the policy."""
    def __init__(self): self.pol, self.ks, self.audit = {}, {}, []
    def grant(self, pid, entity, role, resource, access, t_e):
        ks = self.ks.setdefault(pid, get_random_bytes(32))             # key never leaves the patient device
        pi = f"{entity}|{role}|{resource}|{access}|{t_e}".encode()     # Eq. 8
        p = hmac.new(ks, pi, hashlib.sha256).digest()                  # Eq. 9
        self.pol.setdefault(pid, []).append(dict(e=entity, p=p, a=access, te=t_e))   # ASSUMPTION: entity id + t_e kept alongside p so Eq. 10 can be evaluated
        self.audit.append(("policy_grant", pid, entity)); return p
    def revoke(self, pid, entity, now=None):                           # replaced by pi' with an already-expired t_e
        self.pol[pid] = [x for x in self.pol.get(pid, []) if x["e"] != entity]
        self.pol[pid].append(dict(e=entity, p=b"", a="none", te=(now or time.time()))); self.audit.append(("policy_revoke", pid, entity))
    def check(self, pid, entity, p_submitted, now=None):
        now = now or time.time(); cand = [x for x in self.pol.get(pid, []) if x["e"] == entity]
        chosen = max(cand, key=lambda x: x["te"]) if cand else None    # Eq. 10: pi* = argmax t_e
        ok = bool(chosen) and chosen["te"] > now and hmac.compare_digest(chosen["p"], p_submitted)
        self.audit.append(("policy_allow" if ok else "policy_deny", pid, entity)); return ok

ROLE_PERMS = {"doctor": {"read", "write"}, "nurse": {"read"}, "auditor": {"audit"}}
def rbac_check(role, action): return action in ROLE_PERMS.get(role, ())     # baseline

# --- correctness checks ---
ac = HashCommitmentAC(); s = ac.enroll("p1", "read_ehr"); r = ac.make_request("p1", s)
assert ac.verify(r) and not ac.verify(r)                                           # replayed nonce rejected
assert not ac.verify(dict(ac.make_request("p1", s), ts=time.time() - 3600))        # expired timestamp rejected
assert ac.verify_with_retry("p1", [b"bad1", b"bad2", s], k=3) == 1 and ac.verify_with_retry("p1", [b"a", b"b", b"c", s], k=3) == 0
s2 = ac.revoke_and_reissue("p1", "read_ehr"); assert not ac.verify(ac.make_request("p1", s)) and ac.verify(ac.make_request("p1", s2))
ps = PolicyStore(); t = time.time()
tok_old = ps.grant("p1", "hospA", "doctor", "EHR", "read", t + 100); tok_new = ps.grant("p1", "hospA", "doctor", "EHR", "write", t + 500)
assert ps.check("p1", "hospA", tok_new) and not ps.check("p1", "hospA", tok_old)    # latest expiry wins (Eq. 10)
ps.revoke("p1", "hospA"); assert not ps.check("p1", "hospA", tok_new)
print("Access-control checks passed (replay, expiry, retry limit, revocation, policy conflict resolution).")

# --- Fig 3: verification time, 100 rounds ---
def time_it(fn, n):
    out = []
    for _ in range(n): t0 = time.perf_counter_ns(); fn(); out.append((time.perf_counter_ns() - t0) / 1e6)
    return np.array(out)
N = CFG["AUTH_ROUNDS"]; ac = HashCommitmentAC(); s = ac.enroll("p1", "read_ehr")
reqs = iter([ac.make_request("p1", s) for _ in range(N)])
t_auth = time_it(lambda: ac.verify(next(reqs)), N); t_rbac = time_it(lambda: rbac_check("doctor", "read"), N)
display(pd.DataFrame({"mean_ms": [t_auth.mean(), t_rbac.mean()]}, index=["Hash-commitment authorization", "RBAC baseline"]).round(5))
RESULTS.update(auth_ms=t_auth.mean(), rbac_ms=t_rbac.mean())
fig, ax = plt.subplots(figsize=(7, 3.5)); ax.plot(t_auth, "b--", label="Authorization proof"); ax.plot(t_rbac, "y-", label="RBAC")
ax.set_xlabel("Access attempt"); ax.set_ylabel("Time (ms)"); ax.legend(); ax.grid(alpha=.3); ax.set_title("Access-control verification time (cf. Fig 3)"); plt.show()

## 4. Raft consensus (Sec. 4.4, Algorithm 5, Eq. 11, Sec. 5.5, 6.2-6.4, 6.10)
**(a)** An executable Raft cluster: random leader selection among non-faulty nodes, proposal broadcast, commit when `|Q| ≥ ⌊N/2⌋+1`, crash faults.
**(b)** A message-cost model for latency/throughput: `latency = hops × delay + messages × per-message cost` (**ASSUMPTION**: the paper gives no formula or code).
Message counts follow the paper's complexity statements: Raft O(N), PBFT O(N²), HotStuff/Tendermint leader-based (linear).
Raft modes from Sec. 6.2: Normal, Failover (re-election every 10 rounds), Congestion (extra message delay).

In [ ]:
class RaftCluster:
    def __init__(self, n, seed=0):
        self.n, self.term, self.leader, self.rnd = n, 0, None, random.Random(seed)
        self.alive, self.logs = [True] * n, [[] for _ in range(n)]
    @property
    def quorum(self): return self.n // 2 + 1                          # Eq. 11
    def crash(self, k):
        for i in self.rnd.sample(range(self.n), k):
            self.alive[i] = False
            if i == self.leader: self.leader = None
    def elect(self):                                                    # Alg. 5 line 1: L ~ U(N \ F)
        healthy = [i for i in range(self.n) if self.alive[i]]
        if len(healthy) < self.quorum: self.leader = None; return None
        self.term += 1; self.leader = self.rnd.choice(healthy); return self.leader
    def replicate(self, block):                                         # Alg. 5 lines 2-10
        if self.leader is None or not self.alive[self.leader]:
            if self.elect() is None: return False, 0
        Q, msgs = {self.leader}, 0
        for i in range(self.n):
            if i == self.leader: continue
            msgs += 1
            if self.alive[i]: Q.add(i); msgs += 1; self.logs[i].append(block)
        if len(Q) >= self.quorum: self.logs[self.leader].append(block); return True, msgs
        return False, msgs

for n, k in [(9, 4), (9, 5), (1000, 499), (1000, 500)]:
    c = RaftCluster(n, seed=1); c.crash(k); ok, _ = c.replicate({"tx": 1})
    print(f"Raft N={n:4d}, faulty={k:3d} -> {'committed' if ok else 'no quorum, NULL'}")

def protocol_cost(proto, n, mode="normal", d=CFG["HOP_DELAY_MS"], c=CFG["MSG_COST_MS"]):
    if proto == "Raft":
        hops, msgs = 2, 2 * (n - 1)
        if mode == "failover":   msgs += 2 * (n - 1) / 10                # leader re-election every 10 rounds
        if mode == "congestion": d = d * 3                              # ASSUMPTION: artificial delay = 3x hop delay
    elif proto == "HotStuff":    hops, msgs = 6, 6 * (n - 1)            # 3 leader-based phases
    elif proto == "Tendermint":  hops, msgs = 5, 5 * (n - 1)            # propose + 2 leader-aggregated vote rounds (ASSUMPTION)
    elif proto == "PBFT":        hops, msgs = 3, (n - 1) + (n - 1) ** 2 + n * (n - 1)   # pre-prepare, prepare, commit
    return msgs, hops * d + msgs * c

cases = [("Raft", "normal"), ("Raft", "failover"), ("Raft", "congestion"), ("PBFT", "normal"), ("HotStuff", "normal"), ("Tendermint", "normal")]
rows = []
for proto, mode in cases:
    for n in sorted(set(CFG["RAFT_SIZES"] + CFG["PBFT_SIZES"])):
        msgs, lat = protocol_cost(proto, n, mode)
        rows.append(dict(protocol=proto if mode == "normal" else f"{proto} ({mode})", nodes=n, messages=round(msgs),
                         latency_ms=lat, tps=CFG["BLOCK_SIZE"] / (lat / 1000)))
cons_df = pd.DataFrame(rows)
display(cons_df[cons_df.nodes.isin([20, 100, 1000])].pivot(index="protocol", columns="nodes", values=["messages", "latency_ms", "tps"]).round(1))
get = lambda p, n, col: cons_df.query("protocol==@p and nodes==@n")[col].iloc[0]
RESULTS.update(raft_1000=get("Raft", 1000, "latency_ms"), raft_fo=get("Raft (failover)", 1000, "latency_ms"), raft_cg=get("Raft (congestion)", 1000, "latency_ms"),
               pbft_1000=get("PBFT", 1000, "latency_ms"), raft_tps=get("Raft", 1000, "tps"), pbft_tps=get("PBFT", 1000, "tps"))
fig, ax = plt.subplots(figsize=(7, 4))
for p, g in cons_df.groupby("protocol"): ax.plot(g.nodes, g.latency_ms, marker="o", label=p)
ax.set_xscale("log"); ax.set_yscale("log"); ax.set_xlabel("Nodes"); ax.set_ylabel("Latency (ms, model)"); ax.legend(fontsize=7); ax.grid(alpha=.3, which="both")
ax.set_title("Consensus latency vs. nodes (cf. Figs 4-5)"); plt.show()

## 5. Blockchain layer, data flow, energy (Sec. 3.2, 4.5, 4.6)

In [ ]:
@dataclass
class Block:                                                            # Eq. 12: B_i = <H(B_{i-1}), {c_j}, {p_j}, timestamp, metadata>
    index: int; prev_hash: str; ciphertexts: list; proofs: list; timestamp: float; metadata: dict
    def hash(self):
        h = hashlib.sha256(); h.update(f"{self.index}|{self.prev_hash}|{self.timestamp}".encode())
        for c in self.ciphertexts: h.update(c)
        for p in self.proofs: h.update(p)
        h.update(json.dumps(self.metadata, sort_keys=True).encode()); return h.hexdigest()

class Chain:
    def __init__(self): self.blocks = [Block(0, "0" * 64, [], [], 0.0, {"genesis": True})]
    def append(self, cts, proofs, meta):
        b = Block(len(self.blocks), self.blocks[-1].hash(), cts, proofs, time.time(), meta); self.blocks.append(b); return b
    def verify(self): return all(self.blocks[i].prev_hash == self.blocks[i - 1].hash() for i in range(1, len(self.blocks)))

def run_pipeline(n_patients=40, block_size=10, n_raft=7, n_crashed=2):
    scheme, ac, chain = MockKyberHybrid(), HashCommitmentAC(), Chain()
    raft = RaftCluster(n_raft, seed=3); raft.crash(n_crashed); raft.elect()
    # Data flow steps 1-3: sensors -> redundancy correction -> encryption
    x, _ = inject_faults(generate_signals(n_patients, 20, rng), 5, 0.2, rng); hr, _ = algorithm1(x)
    secrets, pending, where = {}, [], {}
    for i in range(n_patients):
        pid = f"pseudo-{i:04d}"; secrets[pid] = ac.enroll(pid, "read_ehr")
        pending.append((pid, scheme.encrypt(json.dumps({"pid": pid, "hr": round(float(hr[i, -1]), 1)}).encode()), ac.commit[pid]))
    for b in range(0, n_patients, block_size):                          # steps 5-6: leader proposes, quorum commits to the ledger
        chunk = pending[b:b + block_size]; ok, _ = raft.replicate({"blk": b}); assert ok
        blk = chain.append([c for _, c, _ in chunk], [p for _, _, p in chunk], {"n_tx": len(chunk), "term": raft.term})
        for j, (pid, _, _) in enumerate(chunk): where[pid] = (blk.index, j)
    print(f"{len(chain.blocks)-1} blocks committed with {n_crashed}/{n_raft} Raft nodes down; chain valid = {chain.verify()}")
    def request_access(pid, s):                                         # steps 4-7: proof valid AND consensus reached -> grant
        req = ac.make_request(pid, s)
        if not ac.verify(req): return None
        ok, _ = raft.replicate({"access": pid})
        if not ok: return None
        chain.append([], [H(s)], {"event": "access", "pid": pid})       # access logged immutably for audit
        bi, j = where[pid]; return scheme.decrypt(chain.blocks[bi].ciphertexts[j]).decode()
    print("Access with valid proof   ->", request_access("pseudo-0003", secrets["pseudo-0003"]))
    print("Access with wrong secret  ->", request_access("pseudo-0003", b"x" * 32))
    chain.blocks[1].ciphertexts[0] = b"tampered"; print("Chain valid after tampering:", chain.verify())

run_pipeline()

# --- Sec 4.6, Eq. 13: E_tx = n_enc*e_enc + n_cons*e_cons + n_re*e_re ---
def energy_per_tx(n_enc, cons_msgs_per_tx, n_re):
    return (n_enc * CFG["E_CRYPTO_UJ"] + cons_msgs_per_tx * CFG["E_MSG_UJ"] + n_re * CFG["E_RETX_UJ"])
B, N20 = CFG["BLOCK_SIZE"], 20
base = energy_per_tx(1, protocol_cost("PBFT", N20)[0] / B, 0.05)        # ASSUMPTION: retransmission rates 5% (baseline) vs 2% (fault-corrected)
prop = energy_per_tx(1, protocol_cost("Raft", N20)[0] / B, 0.02)
print(f"Energy/tx  RSA+PBFT: {base:.1f} µJ | Raft+Kyber-inspired: {prop:.1f} µJ | reduction {100*(1-prop/base):.1f}%")
RESULTS["energy"] = 100 * (1 - prop / base)

## Paper's reported values vs. this run

In [ ]:
claims = pd.DataFrame([
    ("Encryption time reduction vs RSA-2048", "72.3% (abstract), 91.3% (Sec 6.5)", f"{RESULTS['enc_red']:.1f}%"),
    ("Decryption time reduction vs RSA-2048", "74.4% (abstract), 97.5% (Sec 6.5)", f"{RESULTS['dec_red']:.1f}%"),
    ("Ciphertext size (Kyber-inspired vs RSA)", "32 B vs 256 B (Fig 8); 2400 vs 2800 B (Sec 6.7)", f"{RESULTS['ct_mock']:.0f} B vs {RESULTS['ct_rsa']:.0f} B"),
    ("Peak memory during encryption", "~14% lower", f"{RESULTS['mem_red']:.1f}% lower"),
    ("Authorization verification time", "~0.002 ms (Sec 6.1); <0.4 ms (Sec 3.4/4.3)", f"{RESULTS['auth_ms']:.4f} ms (RBAC {RESULTS['rbac_ms']:.5f} ms)"),
    ("Raft latency @1000 nodes (normal/failover/congestion)", "<115 / 145 / 125 ms (Sec 6.2)", f"{RESULTS['raft_1000']:.0f} / {RESULTS['raft_fo']:.0f} / {RESULTS['raft_cg']:.0f} ms (model)"),
    ("PBFT latency @1000 nodes", ">2000 ms (extrapolated from literature)", f"{RESULTS['pbft_1000']:.0f} ms (model)"),
    ("Raft / PBFT throughput @1000 nodes", "1.26 M / 1.08 M TPS", f"{RESULTS['raft_tps']:.0f} / {RESULTS['pbft_tps']:.1f} TPS (model, block={CFG['BLOCK_SIZE']})"),
    ("Median-filter accuracy @30% faults", ">96% (Sec 6.8); 89.4-93.2% (Sec 6.9)", f"{RESULTS['median_30']:.1f}%"),
    ("Kalman accuracy @30% faults", "92.1-96.7% across rates", f"{RESULTS['kalman_30']:.1f}%"),
    ("Energy reduction vs RSA+PBFT", "9.4-12.3%", f"{RESULTS['energy']:.1f}%"),
], columns=["Metric", "Paper", "This run"])
display(claims.style.hide(axis="index"))

### Assumptions (the paper gives no value or formula)
Fraction of faults that are dropouts (30%) · outlier tolerance ε = 1.5 · true-signal model · Kalman parameters/gating ·
mock sk = pk · version counter in the secret derivation so re-issued tokens differ · entity id and `t_e` stored beside each policy commitment (needed for Eq. 10) ·
network delay, per-message cost and the congestion factor of the consensus cost model · HotStuff/Tendermint message counts ·
transactions per block (20) · retransmission rates and energy per retransmission in Eq. 13.

### Things the paper states inconsistently (the code follows one reading)
Quorum is ⌊N/2⌋+1 in Eq. 11 but ⌈N/2⌉+1 in Lemma 2 / Table 2 (code: ⌊N/2⌋+1) · Sec. 5.3 lists a 15% fault rate that Table 2 and the figures omit (code: 5/10/20/30%) ·
several headline numbers differ between abstract and body (see the table above), so no implementation can match all of them.